# IMPORT LIBRARIES & LOAD DATASET

In [5]:
import pandas as pd
import numpy as np
from sklearn import datasets, linear_model
from sklearn.metrics import mean_squared_error, r2_score
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import statsmodels.api as sm
import statsmodels.formula.api as smf

In [7]:
womens_world_cup_stats = pd.read_csv(
    '/Users/gui/womens_world_cup_2023_statsbomb_finalfinal.csv'
)

In [25]:
womens_world_cup_stats.columns

Index(['match_id', 'team', 'match_date', 'opponent', 'stage', 'goals_for',
       'goals_against', 'result', 'points', 'possession_pct', 'passes',
       'pass_accuracy_pct', 'shots', 'shots_on_target', 'shot_accuracy_pct',
       'xG', 'xG_per_shot', 'goals_minus_xG', 'conversion_pct', 'big_chances',
       'key_passes', 'corners', 'crosses', 'progressive_passes',
       'progressive_carries', 'final_third_entries', 'box_entries',
       'high_turnovers', 'shots_after_turnover', 'goals_after_turnover',
       'set_piece_goals', 'tackles', 'tackles_won', 'interceptions', 'blocks',
       'clearances', 'recoveries', 'saves', 'clean_sheet', 'formation',
       'shot_assists', 'through_balls', 'cutbacks', 'dribbles',
       'successful_dribbles', 'dribble_success_pct', 'duels', 'duels_won',
       'counterpress_actions', 'goals_from_shots', 'own_goals_for',
       'own_goals_against', 'goal_type', 'win'],
      dtype='object')

# Create win variable 

In [14]:
womens_world_cup_stats["win"] = (
    womens_world_cup_stats["goals_for"] >
    womens_world_cup_stats["goals_against"]
).astype(int)

## Corelation between possession & victory

In [16]:
possession_win_corr = womens_world_cup_stats[
    ["possession_pct", "win"]
].corr().iloc[0, 1]

print(f"Correlation between possession and winning: {possession_win_corr:.3f}")

Correlation between possession and winning: 0.422


## Corelation between goals & shots

In [21]:
shots_goals_corr = womens_world_cup_stats[
    ["shots", "goals_for"]
].corr().iloc[0, 1]

print(f"Correlation between possession and winning: {shots_goals_corr:.3f}")

Correlation between possession and winning: 0.525


In [33]:
xg_goals_corr = womens_world_cup_stats[
    ["xG", "goals_for"]
].corr().iloc[0, 1]

print(f"Correlation between goals and xG: {xg_goals_corr:.3f}")

Correlation between goals and xG: 0.340


## Team effiency 

In [36]:
team_efficiency = womens_world_cup_stats.groupby("team").agg(
    goals=("goals_for", "sum"),
    xG=("xG", "sum")
).reset_index()

team_efficiency["goals_minus_xG"] = (
    team_efficiency["goals"] - team_efficiency["xG"]
)

team_efficiency.sort_values(
    "goals_minus_xG", ascending=False
).head(10)

,team,goals,xG,goals_minus_xG
14,Japan Women's,15,10.685346,4.314654
17,Netherlands Women's,12,8.284816,3.715184
10,Germany Women's,8,5.128810,2.871190
5,Colombia Women's,6,4.140556,1.859444
25,South Africa Women's,6,4.942529,1.057471
21,Panama Women's,3,2.097455,0.902545
0,Argentina Women's,2,1.164784,0.835216
20,Norway Women's,7,6.296983,0.703017
31,Zambia W,3,2.354450,0.645550
22,Philippines Women's,1,0.590308,0.409692


In [38]:
team_efficiency.sort_values("goals", ascending=False)

,team,goals,xG,goals_minus_xG
26,Spain Women's,18,18.187878,-0.187878
14,Japan Women's,15,10.685346,4.314654
27,Sweden Women's,14,17.209317,-3.209317
8,England Women's,13,13.520331,-0.520331
17,Netherlands Women's,12,8.284816,3.715184
9,France Women's,12,17.485259,-5.485259
1,Australia Women's,10,19.375918,-9.375918
10,Germany Women's,8,5.128810,2.871190
20,Norway Women's,7,6.296983,0.703017
5,Colombia Women's,6,4.140556,1.859444


In [46]:
team_efficiency = womens_world_cup_stats.groupby("team").agg(
    goals=("goals_for", "sum"),
    xG=("xG", "sum")
).reset_index()

team_efficiency["goals_minus_xG"] = (
    team_efficiency["goals"] - team_efficiency["xG"]
)

team_efficiency.sort_values(
    "goals_minus_xG", ascending=True
).head(10)

,team,goals,xG,goals_minus_xG
1,Australia Women's,10,19.375918,-9.375918
29,United States Women's,4,13.366425,-9.366425
9,France Women's,12,17.485259,-5.485259
19,Nigeria Women's,3,7.006399,-4.006399
27,Sweden Women's,14,17.209317,-3.209317
18,New Zealand Women's,1,3.685054,-2.685054
23,Portugal Women's,2,4.231031,-2.231031
11,Haiti Women's,0,1.935472,-1.935472
3,Canada Women's,2,3.907472,-1.907472
15,Korea Republic Women's,1,2.726093,-1.726093


## Defensive efficiency

In [56]:
defensive_efficiency = womens_world_cup_stats.groupby("team").agg(
    saves=("saves", "sum"),
    tackles=("tackles", "sum"),
    tackles_won=("tackles_won", "sum"),
    interceptions=("interceptions", "sum"),
    blocks=("blocks", "sum"),
    clearances=("clearances", "sum"),
    duels=("duels", "sum"),
    duels_won=("duels_won", "sum"),
    clean_sheets=("clean_sheet", "sum"),
    goals_conceded=("goals_against", "sum")
).reset_index()

In [58]:
defensive_efficiency["goals_conceded_per_clean_sheet"] = (
    defensive_efficiency["goals_conceded"] /
    defensive_efficiency["clean_sheets"].replace(0, 1)
)

In [62]:
defensive_efficiency

,team,saves,tackles,tackles_won,interceptions,blocks,clearances,duels,duels_won,clean_sheets,goals_conceded,goals_conceded_per_clean_sheet
0,Argentina Women's,6,87,45.0,33,67,70,123,48,0,5,5.000000
1,Australia Women's,23,124,76.0,60,156,203,256,79,4,8,2.000000
2,Brazil Women's,7,66,43.0,23,73,63,108,44,2,2,1.000000
3,Canada Women's,9,62,35.0,30,76,64,118,40,1,5,5.000000
4,China PR Women's,3,72,42.0,37,73,118,133,44,1,7,7.000000
5,Colombia Women's,15,137,80.0,53,128,137,221,81,2,4,2.000000
6,Costa Rica Women's,24,53,26.0,36,85,120,109,26,0,8,8.000000
7,Denmark Women's,8,92,49.0,29,108,113,171,54,2,3,1.500000
8,England Women's,17,153,77.0,62,156,190,284,86,3,4,1.333333
9,France Women's,8,140,86.0,59,124,109,232,89,3,4,1.333333


In [68]:
# Defensive metrics per game
defensive_efficiency["tackles_won_pct"] = (
    defensive_efficiency["tackles_won"] /
    defensive_efficiency["tackles"] * 100
)

defensive_efficiency["duels_won_pct"] = (
    defensive_efficiency["duels_won"] /
    defensive_efficiency["duels"] * 100
)

defensive_efficiency["blocks_per_game"] = (
    defensive_efficiency["blocks"] /
    defensive_efficiency["matches_played"]
)

defensive_efficiency["interceptions_per_game"] = (
    defensive_efficiency["interceptions"] /
    defensive_efficiency["matches_played"]
)

defensive_efficiency["clean_sheet_rate"] = (
    defensive_efficiency["clean_sheets"] /
    defensive_efficiency["matches_played"] * 100
)

defensive_efficiency["goals_conceded_per_game"] = (
    defensive_efficiency["goals_conceded"] /
    defensive_efficiency["matches_played"]
)


# Min-Max scaling
def min_max_score(series):
    return (series - series.min()) / (series.max() - series.min())


# Positive metrics: higher = better
defensive_efficiency["clean_sheet_score"] = min_max_score(
    defensive_efficiency["clean_sheet_rate"]
)

defensive_efficiency["tackles_won_score"] = min_max_score(
    defensive_efficiency["tackles_won_pct"]
)

defensive_efficiency["duels_won_score"] = min_max_score(
    defensive_efficiency["duels_won_pct"]
)

defensive_efficiency["blocks_score"] = min_max_score(
    defensive_efficiency["blocks_per_game"]
)

defensive_efficiency["interceptions_score"] = min_max_score(
    defensive_efficiency["interceptions_per_game"]
)


# Goals conceded: lower = better, so invert the score
defensive_efficiency["goals_conceded_score"] = (
    1 - min_max_score(
        defensive_efficiency["goals_conceded_per_game"]
    )
)


# Defensive Efficiency Score
defensive_efficiency["defensive_efficiency_score"] = (
    defensive_efficiency["goals_conceded_score"] * 0.30 +
    defensive_efficiency["clean_sheet_score"] * 0.25 +
    defensive_efficiency["tackles_won_score"] * 0.15 +
    defensive_efficiency["duels_won_score"] * 0.15 +
    defensive_efficiency["blocks_score"] * 0.10 +
    defensive_efficiency["interceptions_score"] * 0.05
) * 100


# Final ranking: best → worst
defensive_ranking = defensive_efficiency.sort_values(
    "defensive_efficiency_score",
    ascending=False
)[[
    "team",
    "matches_played",
    "goals_conceded_per_game",
    "clean_sheet_rate",
    "tackles_won_pct",
    "duels_won_pct",
    "blocks_per_game",
    "interceptions_per_game",
    "defensive_efficiency_score"
]].reset_index(drop=True)

defensive_ranking.index = defensive_ranking.index + 1

defensive_ranking

,team,matches_played,goals_conceded_per_game,clean_sheet_rate,tackles_won_pct,duels_won_pct,blocks_per_game,interceptions_per_game,defensive_efficiency_score
1,Nigeria Women's,4,0.500000,75.000000,60.439560,33.734940,26.500000,12.000000,84.679483
2,United States Women's,4,0.250000,75.000000,59.302326,36.241611,23.750000,11.000000,84.222733
3,Netherlands Women's,5,0.600000,60.000000,59.333333,43.243243,26.400000,12.600000,83.976920
4,Brazil Women's,3,0.666667,66.666667,65.151515,40.740741,24.333333,7.666667,83.114207
5,France Women's,5,0.800000,60.000000,61.428571,38.362069,24.800000,11.800000,78.984289
6,Portugal Women's,3,0.333333,66.666667,56.521739,33.620690,25.000000,11.333333,78.699602
7,Sweden Women's,7,0.571429,57.142857,63.698630,36.090226,23.000000,9.857143,77.192925
8,Jamaica Women's,4,0.250000,75.000000,49.473684,27.976190,25.000000,12.500000,74.786039
9,Japan Women's,5,0.600000,60.000000,53.684211,33.742331,20.400000,13.400000,69.629301
10,Colombia Women's,5,0.800000,40.000000,58.394161,36.651584,25.600000,10.600000,69.199345


## Corelation between key passes & goals

In [78]:
shot_t_goals_corr = womens_world_cup_stats[
    ["shots_on_target", "goals_for"]
].corr().iloc[0, 1]

print(f"Correlation between shots on target and goals: {shot_t_goals_corr:.3f}")

Correlation between shots on target and goals: 0.570


## Corelation between shot assists & goals

In [76]:
shotassists_goals_corr = womens_world_cup_stats[
    ["shot_assists", "goals_for"]
].corr().iloc[0, 1]

print(f"Correlation between shot assists and goals: {shotassists_goals_corr:.3f}")

Correlation between shot assists and goals: 0.467


In [84]:
trans_efficiency = womens_world_cup_stats.groupby("team").agg(
    shots_after_turnover=("shots_after_turnover", "sum"),
    goals_after_turnover=("goals_after_turnover", "sum")
).reset_index()

trans_efficiency["transition_efficiency_pct"] = (
    trans_efficiency["goals_after_turnover"] /
    trans_efficiency["shots_after_turnover"] * 100
)

trans_efficiency = trans_efficiency.sort_values(
    "transition_efficiency_pct",
    ascending=False
)

trans_efficiency.head(10)

,team,shots_after_turnover,goals_after_turnover,transition_efficiency_pct
27,Sweden Women's,24.0,10.0,41.666667
17,Netherlands Women's,14.0,5.0,35.714286
26,Spain Women's,42.0,10.0,23.809524
20,Norway Women's,9.0,2.0,22.222222
9,France Women's,20.0,4.0,20.000000
14,Japan Women's,16.0,2.0,12.500000
1,Australia Women's,25.0,3.0,12.000000
8,England Women's,21.0,2.0,9.523810
10,Germany Women's,11.0,1.0,9.090909
19,Nigeria Women's,12.0,1.0,8.333333
